# Notebook 14 — Create Paper Tables and Figures

## Goal
Generate all publication-ready tables and figures for the paper.
Every table and figure is created here directly from approved artifacts — no manual editing.

---

## Outputs created
- Table 1: Data summary
- Table 2: Feature summary
- Table 3: Model comparison (main results)
- Table 4: Worst forecast errors
- Table 5: RAG example (one forecast with evidence + explanation)
- Table 6: RAG evaluation summary
- Figure 1: Actual vs. predicted (macro+news)
- Figure 2: News trend features over time
- Figure 3: Feature importance
- Figure 4: Error over time

---

## What you must inspect
- All tables have correct column labels and units
- Figures have axis labels, titles, and legends
- Metrics match those reported in Notebook 09

In [ ]:
import pathlib, json, datetime
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import joblib
import yaml

DRIVE_ROOT = pathlib.Path('/content/drive/MyDrive/labor_news_rag_project')
REPO_DIR = pathlib.Path('/content/economic-news-labor-rag')

REPORTS_TABLES = DRIVE_ROOT / 'outputs' / 'tables'
REPORTS_FIGS = DRIVE_ROOT / 'outputs' / 'figures'
REPORTS_EXAMPLES = DRIVE_ROOT / 'outputs' / 'explanations'

with open(REPO_DIR / 'configs' / 'model_config.yaml') as f:
    model_cfg = yaml.safe_load(f)

TARGET = model_cfg['target_column']
TEST_START = pd.Timestamp(model_cfg['splits']['test_start'])
TRAIN_END = pd.Timestamp(model_cfg['splits']['train_end'])
VAL_END = pd.Timestamp(model_cfg['splits']['validation_end'])

# Load artifacts
model_df = pd.read_parquet(DRIVE_ROOT / 'data' / 'model_ready' / 'labor_forecasting_dataset_approved.parquet')
model_df['forecast_date'] = pd.to_datetime(model_df['forecast_date'])

news_features = pd.read_parquet(DRIVE_ROOT / 'data' / 'features' / 'news' / 'monthly_news_features_approved.parquet')

macro_preds = pd.read_parquet(DRIVE_ROOT / 'outputs' / 'predictions' / 'predictions_macro_only.parquet')
mn_preds = pd.read_parquet(DRIVE_ROOT / 'outputs' / 'predictions' / 'predictions_macro_news.parquet')
naive_preds = pd.read_parquet(DRIVE_ROOT / 'outputs' / 'predictions' / 'predictions_naive.parquet')

for df in [macro_preds, mn_preds, naive_preds]:
    df['forecast_date'] = pd.to_datetime(df['forecast_date'])
    df['error'] = df['actual'] - df['prediction']

baseline_metrics = pd.read_csv(DRIVE_ROOT / 'outputs' / 'metrics' / 'baseline_metrics.csv')
model_comparison = pd.read_csv(DRIVE_ROOT / 'outputs' / 'metrics' / 'model_comparison.csv')

explanations_df = pd.read_parquet(DRIVE_ROOT / 'outputs' / 'explanations' / 'forecast_explanations.parquet')
evidence_df = pd.read_parquet(DRIVE_ROOT / 'outputs' / 'explanations' / 'retrieved_evidence.parquet')
evidence_df['forecast_date'] = pd.to_datetime(evidence_df['forecast_date'])
evidence_df['article_date'] = pd.to_datetime(evidence_df['article_date'])

print('All artifacts loaded.')

## Table 1 — Data Summary

In [ ]:
clean_news = pd.read_parquet(DRIVE_ROOT / 'data' / 'interim' / 'news' / 'clean_news_approved.parquet')

table1 = pd.DataFrame([
    {'Variable': 'Nonfarm Payrolls (PAYEMS)', 'Source': 'FRED/ALFRED', 'Freq': 'Monthly',
     'Start': '2000-01', 'End': '2024-12', 'N': model_df['forecast_date'].nunique()},
    {'Variable': 'Unemployment Rate', 'Source': 'FRED/ALFRED', 'Freq': 'Monthly',
     'Start': '2000-01', 'End': '2024-12', 'N': model_df['unrate_lag1'].notna().sum()},
    {'Variable': 'Initial Claims (ICSA)', 'Source': 'FRED/ALFRED', 'Freq': 'Weekly→Monthly',
     'Start': '2000-01', 'End': '2024-12', 'N': model_df['icsa_lag1'].notna().sum()},
    {'Variable': 'Job Openings (JTSJOL)', 'Source': 'FRED/ALFRED', 'Freq': 'Monthly',
     'Start': '2001-01', 'End': '2024-12', 'N': model_df['jtsjol_lag1'].notna().sum()},
    {'Variable': 'GDELT News Articles', 'Source': 'GDELT DOC 2.0', 'Freq': 'Monthly aggregated',
     'Start': '2000-01', 'End': '2024-12', 'N': len(clean_news)},
])

print('=== Table 1: Data Summary ===')
print(table1.to_string(index=False))
table1.to_csv(REPORTS_TABLES / 'table1_data_summary.csv', index=False)
print('Saved: table1_data_summary.csv')

## Table 3 — Main Model Comparison (key paper table)

In [ ]:
from sklearn.metrics import mean_absolute_error, mean_squared_error

def calc_metrics(pred_df):
    a = pred_df['actual'].values
    p = pred_df['prediction'].values
    mask = ~(np.isnan(a) | np.isnan(p))
    a, p = a[mask], p[mask]
    return {
        'N': int(mask.sum()),
        'MAE (K)': round(float(mean_absolute_error(a, p)), 1),
        'RMSE (K)': round(float(np.sqrt(mean_squared_error(a, p))), 1),
        'DA (%)': round(float(np.mean(np.sign(a) == np.sign(p)) * 100), 1),
    }

# Load news and arima predictions
try:
    arima_preds = pd.read_parquet(DRIVE_ROOT / 'outputs' / 'predictions' / 'predictions_arima.parquet')
    arima_preds['forecast_date'] = pd.to_datetime(arima_preds['forecast_date'])
    arima_preds['error'] = arima_preds['actual'] - arima_preds['prediction']
    news_preds = pd.read_parquet(DRIVE_ROOT / 'outputs' / 'predictions' / 'predictions_news_only.parquet')
    news_preds['forecast_date'] = pd.to_datetime(news_preds['forecast_date'])
    has_all_preds = True
except FileNotFoundError:
    has_all_preds = False
    print('Note: Some prediction files not found. Using available files.')

table3_rows = [
    {'Model': 'Naive Baseline', **calc_metrics(naive_preds)},
    {'Model': 'XGBoost (Macro-Only)', **calc_metrics(macro_preds)},
    {'Model': 'XGBoost (Macro+News)', **calc_metrics(mn_preds)},
]
if has_all_preds:
    table3_rows.insert(1, {'Model': 'ARIMA', **calc_metrics(arima_preds)})
    table3_rows.insert(3, {'Model': 'XGBoost (News-Only)', **calc_metrics(news_preds)})

table3 = pd.DataFrame(table3_rows)
print('=== Table 3: Model Comparison (Test Set) ===')
print(table3.to_string(index=False))
table3.to_csv(REPORTS_TABLES / 'table3_model_comparison.csv', index=False)
print('\nSaved: table3_model_comparison.csv')

# Highlight main finding
macro_mae = table3[table3['Model'] == 'XGBoost (Macro-Only)']['MAE (K)'].iloc[0]
mn_mae = table3[table3['Model'] == 'XGBoost (Macro+News)']['MAE (K)'].iloc[0]
if mn_mae < macro_mae:
    print(f'\nFINDING: Macro+News improves over Macro-Only by {(macro_mae - mn_mae):.1f}K MAE ({(macro_mae - mn_mae)/macro_mae*100:.1f}%).')
else:
    print(f'\nFINDING: Macro+News does NOT beat Macro-Only (MAE: {mn_mae:.1f}K vs {macro_mae:.1f}K).')

## Figure 1 — Actual vs. Predicted (paper quality)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 7))

ax.scatter(mn_preds['actual'], mn_preds['prediction'],
           color='steelblue', alpha=0.7, s=50, zorder=3, label='Forecast')

lim = max(abs(mn_preds['actual'].max()), abs(mn_preds['actual'].min())) * 1.15
ax.plot([-lim, lim], [-lim, lim], 'r--', linewidth=1.5, label='45° line (perfect forecast)', zorder=2)
ax.axhline(0, color='gray', linewidth=0.5, alpha=0.5)
ax.axvline(0, color='gray', linewidth=0.5, alpha=0.5)

ax.set_xlabel('Actual Nonfarm Payroll Change (K)', fontsize=12)
ax.set_ylabel('Predicted Nonfarm Payroll Change (K)', fontsize=12)
ax.set_title('Actual vs. Predicted\nXGBoost (Macro + News), Test Set 2022–2024', fontsize=12)
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)

mae = mean_absolute_error(mn_preds['actual'], mn_preds['prediction'])
ax.text(0.05, 0.95, f'MAE = {mae:.1f}K', transform=ax.transAxes,
        fontsize=10, verticalalignment='top',
        bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

plt.tight_layout()
fig_path = REPORTS_FIGS / 'fig1_actual_vs_predicted_paper.png'
plt.savefig(str(fig_path), dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved: {fig_path.name}')

## Figure 2 — News trend features over time

In [ ]:
months_idx = range(len(news_features))

fig, axes = plt.subplots(3, 1, figsize=(12, 10))

axes[0].plot(months_idx, news_features['layoff_news_count'], color='firebrick', label='Layoffs')
axes[0].plot(months_idx, news_features['hiring_news_count'], color='seagreen', label='Hiring')
axes[0].set_title('Labor Market News Signals: Layoffs and Hiring')
axes[0].set_ylabel('Monthly article count')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(months_idx, news_features['uncertainty_news_count'], color='darkorange')
axes[1].set_title('Economic Uncertainty News Count')
axes[1].set_ylabel('Monthly article count')
axes[1].grid(True, alpha=0.3)

axes[2].plot(months_idx, news_features['news_volume'], color='steelblue')
axes[2].set_title('Total Labor Market News Volume')
axes[2].set_ylabel('Monthly article count')
axes[2].set_xlabel('Month index (2000–2024)')
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
fig_path = REPORTS_FIGS / 'fig2_news_trends_paper.png'
plt.savefig(str(fig_path), dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved: {fig_path.name}')

## Figure 3 — Feature importance

In [ ]:
try:
    xgb_mn = joblib.load(DRIVE_ROOT / 'artifacts' / 'models' / 'xgboost_macro_news.joblib')
    available_all = [f for f in model_cfg['macro_features'] + model_cfg['news_features']
                     if f in model_df.columns]
    importances = pd.Series(xgb_mn.feature_importances_, index=available_all).sort_values()

    fig, ax = plt.subplots(figsize=(10, 6))
    news_feat_set = set(model_cfg['news_features'])
    colors = ['darkorange' if f in news_feat_set else 'steelblue' for f in importances.index]
    importances.plot(kind='barh', ax=ax, color=colors, edgecolor='black', linewidth=0.5)
    ax.set_title('Feature Importance — XGBoost (Macro + News)\n(orange = news features, blue = macro features)')
    ax.set_xlabel('Importance score (gain)')
    ax.grid(True, axis='x', alpha=0.3)

    from matplotlib.patches import Patch
    legend_elements = [Patch(facecolor='steelblue', label='Macro features'),
                       Patch(facecolor='darkorange', label='News features')]
    ax.legend(handles=legend_elements, loc='lower right')

    plt.tight_layout()
    fig_path = REPORTS_FIGS / 'fig3_feature_importance_paper.png'
    plt.savefig(str(fig_path), dpi=150, bbox_inches='tight')
    plt.show()
    print(f'Saved: {fig_path.name}')
except FileNotFoundError:
    print('Model file not found. Run Notebook 09 first.')

## Table 5 — RAG Example (one forecast with evidence + explanation)

In [ ]:
# Pick the forecast with the most interesting prediction (e.g., largest absolute prediction)
if len(explanations_df) > 0:
    example_row = explanations_df.iloc[abs(explanations_df['prediction']).argmax()]
    fd = pd.Timestamp(example_row['forecast_date'])
    ev = evidence_df[evidence_df['forecast_date'] == fd].head(5)

    print('=== Table 5: RAG Example ===')
    print(f'Forecast month: {example_row["forecast_month"]}')
    print(f'Predicted: {example_row["prediction"]:+.0f}K | Actual: {example_row["actual"]:+.0f}K')
    print(f'\nRetrieved evidence:')
    ev_table = ev[['article_date', 'query_group', 'title', 'url']].copy()
    ev_table['article_date'] = ev_table['article_date'].dt.date
    ev_table['title'] = ev_table['title'].str[:60]
    ev_table['url'] = ev_table['url'].str[:50]
    print(ev_table.to_string(index=False))
    print(f'\nExplanation:')
    print(example_row['explanation'])

    # Save as CSV
    example_table = pd.DataFrame([{
        'forecast_month': example_row['forecast_month'],
        'prediction_K': round(example_row['prediction'], 0),
        'actual_K': round(example_row['actual'], 0),
        'n_evidence_articles': example_row['n_articles_retrieved'],
        'explanation': example_row['explanation'],
    }])
    example_table.to_csv(REPORTS_TABLES / 'table5_rag_example.csv', index=False)
    ev_table.to_csv(REPORTS_TABLES / 'table5_rag_example_evidence.csv', index=False)
    print('\nSaved: table5_rag_example.csv, table5_rag_example_evidence.csv')
else:
    print('No explanations available. Run NB 12 first.')

## Paper outputs complete

All tables and figures have been saved to:
- `outputs/tables/` — CSV tables
- `outputs/figures/` — PNG figures

Proceed to `15_final_reproducibility_check.ipynb`.